# 04 — Evaluasi (publik) dan foto K919

1. Threshold confidence dipilih di split **val** → diterapkan ke split **test** (evaluasi akhir dataset publik).
2. Prediksi dibuat lewat **adapter API** (`app.inference.UltralyticsDetector`) — keluaran sama dengan `/analyze`.
3. Foto K919: statistik domain (ukuran, orientasi, ketajaman) vs dataset publik, dan — **hanya bila ada label
   manusia** — evaluasi terpisah dengan domain `k919`.

Hasil dataset publik **bukan** kinerja pada foto K919. Confidence model bukan ukuran akurasi survei.

In [ ]:
%matplotlib inline
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display

import wallacea_ml
from wallacea_ml.prepare import ML_ROOT
print("wallacea_ml", wallacea_ml.__version__, "| ML_ROOT =", ML_ROOT)


def _df(rows):
    """Tabel: pandas bila terpasang (opsional), selain itu list apa adanya."""
    try:
        import pandas as pd
        return pd.DataFrame(rows)
    except ImportError:
        return rows


In [ ]:
from wallacea_ml.evaluate import evaluate, sweep, choose_threshold
DATASET = "uavmulticrop_maize"
MODEL = f"{DATASET}_yolo11n"
EVAL_DIR = ML_ROOT / "artifacts" / "eval" / MODEL
card_path = ML_ROOT / "artifacts" / "models" / MODEL / "model_card.json"
pred_val = json.loads((EVAL_DIR / "pred_val.json").read_text(encoding="utf-8"))
pred_test = json.loads((EVAL_DIR / "pred_test.json").read_text(encoding="utf-8"))
rows = sweep(pred_val)
best = choose_threshold(rows, "count_mae")
print("threshold dipilih di VAL:", best)
fig, ax = plt.subplots(1, 2, figsize=(13, 3.5))
c = [r["conf"] for r in rows]
ax[0].plot(c, [r["precision"] for r in rows], label="precision"); ax[0].plot(c, [r["recall"] for r in rows], label="recall")
ax[0].plot(c, [r["f1"] for r in rows], label="F1"); ax[0].axvline(best["conf"], ls="--", c="k"); ax[0].legend(); ax[0].set_xlabel("confidence")
ax[1].plot(c, [r["count_mae_per_image"] for r in rows]); ax[1].axvline(best["conf"], ls="--", c="k"); ax[1].set_title("MAE hitungan per foto (VAL)")
plt.show()

## Evaluasi akhir pada split TEST (threshold dari VAL, tidak diubah)

In [ ]:
r = evaluate(pred_test, leaks=[], conf_threshold=best["conf"])
print({k: getattr(r, k) for k in ("domain", "images", "gt_total", "tp", "fp", "fn", "precision", "recall", "f1", "ap",
                                  "count_mae_per_image", "count_mean_error_per_image", "count_relative_error")})
for cv in r.caveats: print("•", cv)
gt = [row["gt"] for row in r.per_image]; pr = [row["pred"] for row in r.per_image]
plt.figure(figsize=(4.5, 4.5)); plt.scatter(gt, pr); m_ = max(gt + pr) * 1.05
plt.plot([0, m_], [0, m_], "k--"); plt.xlabel("hitungan label manusia"); plt.ylabel("hitungan model"); plt.title("per foto — TEST"); plt.show()

## Visual: prediksi (biru) vs label (hijau) pada foto test

In [ ]:
from wallacea_ml import backend
from wallacea_ml.viz import draw_boxes
ROOT = ML_ROOT / "data" / "prepared" / DATASET
im = max(pred_test["images"], key=lambda i: len(i["gt"]))
boxes = [("plant", g["bbox"]) for g in im["gt"]] + [("pred", p["bbox"]) for p in im["pred"] if p["confidence"] >= best["conf"]]
plt.figure(figsize=(14, 8)); plt.imshow(draw_boxes(backend.load_oriented(ROOT / im["image"]), boxes, max_side=2400)); plt.axis("off"); plt.show()

## Foto K919

Salin foto ASLI dari kartu/aplikasi drone (bukan screenshot) ke `ml/data/k919/raw/` (tidak masuk Git).
Bila tim sudah melabeli sebagian (CVAT → COCO, label `plant`), buat config prepare dengan `domain: k919`
dan grup per lahan/penerbangan, lalu jalankan `prepare` → `split` → `predict` → `evaluate`.

In [ ]:
from wallacea_ml.inspection import photo_folder_stats
K919 = ML_ROOT / "data" / "k919" / "raw"
k_rows = photo_folder_stats(K919) if K919.is_dir() else []
print(len(k_rows), "foto K919 ditemukan" if k_rows else "— belum ada foto K919 di data/k919/raw (status: BELUM DIUJI)")
if k_rows:
    display(_df(k_rows[:20]))
    print("orientasi EXIF:", Counter(r["exif_orientation"] for r in k_rows))

## Perbandingan domain: dataset publik vs K919

In [ ]:
from wallacea_ml.manifest import load_manifest
from wallacea_ml.inspection import inspect_manifest, compare_domains
pub = inspect_manifest(load_manifest(ROOT / "manifest.json"), ROOT, open_images=False).summary()
doms = {"public:" + DATASET: pub}
if k_rows:
    doms["k919 (tanpa label)"] = {"images": len(k_rows), "boxes": None,
                                  "image_sizes": dict(Counter(f"{r['width']}x{r['height']}" for r in k_rows))}
display(_df(compare_domains(doms)))
print("Ukuran objek tanaman pada foto K919 bergantung ketinggian terbang; bandingkan lebar box (px) setelah ada label K919.")

## Prediksi pada foto K919 (tanpa label = BUKAN akurasi)

In [ ]:
if k_rows and card_path.is_file():
    card = json.loads(card_path.read_text(encoding="utf-8"))
    env = card["api_env"]
    det = backend.build_api_detector(env["WALLACEA_MODEL_PATH"], json.loads(env["WALLACEA_LABEL_MAP_JSON"]),
                                     confidence=env["WALLACEA_CONFIDENCE"] or best["conf"], imgsz=env["WALLACEA_IMGSZ"])
    for r_ in k_rows[:3]:
        img = backend.load_oriented(K919 / r_["image"])
        out = det.detect(img)
        plt.figure(figsize=(12, 7)); plt.imshow(draw_boxes(img, [("pred", d.bbox) for d in out.detections], max_side=2400))
        plt.title(f"{r_['image']}: {len(out.detections)} kandidat (pending — perlu tinjauan manusia)"); plt.axis("off"); plt.show()
else:
    print("Lewati: butuh foto K919 dan model card.")